# Phase 120: CryptoBench + B-factor Flexibility Feature

Cryptic pockets are, by definition, regions that are apo-invisible but
become pockets upon binding -- they require local flexibility. B-factors
(crystallographic thermal/disorder factors, free in every PDB file) are a
direct experimental measure of that. Per-chain z-scored and added as one
extra feature column. Ingests fresh (Phase117's blob doesn't have this),
then trains with-vs-without (same architecture, just zeroing the one
column for the ablation) in the same run for direct comparison.

## Cell 1: Install dependencies (identical to Phase117)

In [ ]:
!pip install -q torch-geometric==2.6.1 gudhi==3.10.1 biopython==1.84 transformers==4.46.3

## Cell 2: Setup, fetch CryptoBench's dataset.json + folds.json + noncryptic-pockets.json (identical to Phase117)

In [ ]:
import gzip, json, os, time, urllib.request, warnings
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import gudhi
from scipy.spatial import cKDTree
from torch_geometric.data import Data
from Bio.PDB import MMCIFParser, PDBParser, PPBuilder

warnings.filterwarnings("ignore")

WORK = Path("/kaggle/working"); (WORK/"cache").mkdir(parents=True, exist_ok=True)
RAW = Path("/tmp/pdb_raw"); RAW.mkdir(parents=True, exist_ok=True)

GPU_OK=False
if torch.cuda.is_available():
    try:
        _p=torch.randn(256,256,device="cuda"); (_p@_p).sum().item(); torch.cuda.synchronize(); GPU_OK=True
        print(f"GPU: {torch.cuda.get_device_name(0)} (probe PASS)")
    except RuntimeError as e: print("GPU unusable:", e)
DEVICE = torch.device("cuda" if GPU_OK else "cpu")

def fetch_json(url):
    with urllib.request.urlopen(url, timeout=120) as r:
        return json.loads(r.read().decode())

cb_dataset = fetch_json("https://osf.io/download/ta2ju/")
cb_folds = fetch_json("https://osf.io/download/5s93p/")
cb_noncryptic = fetch_json("https://osf.io/download/37ztm/")
print(f"CryptoBench: {len(cb_dataset):,} apo entries | folds: "
      f"{ {k: len(v) for k,v in cb_folds.items()} } | "
      f"noncryptic-pocket apo entries: {len(cb_noncryptic):,}")

# --- pick the MAIN holo pairing (unique per apo entry, verified: exactly
# one is_main_holo_structure==True per apo_id, 0 zero-main / 0 multi-main) --
units = []
for apo_id, entries in cb_dataset.items():
    main = next((e for e in entries if e.get("is_main_holo_structure")), None)
    if main is None: continue
    # keep as string, not int: residue numbers can carry a PDB insertion
    # code suffix (e.g. "60A"), which int() rejects
    pocket_resnums = {sel.split("_", 1)[1] for sel in main["apo_pocket_selection"]}
    apo_chain = main["apo_chain"]

    # noncryptic-pockets.json is keyed by 4-char PDB ID only; each entry
    # carries its own apo_chain -- filter to entries matching THIS unit's
    # apo_chain, union all their apo_pocket_selection residues. Multiple
    # entries per PDB ID are common (up to ~12, different ligand/holo
    # pairings at the same or nearby sites) -- union captures all of them.
    noncryptic_resnums = set()
    for nc in cb_noncryptic.get(apo_id, []):
        if nc.get("apo_chain") != apo_chain: continue
        noncryptic_resnums |= {sel.split("_", 1)[1] for sel in nc["apo_pocket_selection"]}

    units.append({"apo_id": apo_id, "apo_chain": apo_chain,
                   "pocket_resnums": pocket_resnums,
                   "noncryptic_resnums": noncryptic_resnums, "pRMSD": main.get("pRMSD")})
n_with_noncryptic = sum(1 for u in units if u["noncryptic_resnums"])
print(f"training units (one main pocket per apo entry): {len(units):,} "
      f"({n_with_noncryptic:,} have >=1 noncryptic-pocket residue on the same chain)")

split_of = {}
for pid in cb_folds.get("test", []): split_of[pid] = "test"
for fold_key in ("train-0","train-1","train-2","train-3"):
    for pid in cb_folds.get(fold_key, []): split_of[pid] = "train"
for u in units:
    u["split"] = split_of.get(u["apo_id"], "unassigned")
n_unassigned = sum(1 for u in units if u["split"] == "unassigned")
print(f"split coverage: {sum(1 for u in units if u['split']!='unassigned')}/{len(units)} "
      f"({n_unassigned} unassigned, dropped)")
units = [u for u in units if u["split"] != "unassigned"]

## Cell 3: Fetch apo structures (identical to Phase117)

In [ ]:
need = sorted({u["apo_id"] for u in units})
def fetch(pid):
    for ext in ("cif","pdb"):
        p = RAW / f"{pid}.{ext}.gz"
        if p.exists() and p.stat().st_size > 500: return pid, p
    for ext in ("cif","pdb"):
        try:
            with urllib.request.urlopen(f"https://files.rcsb.org/download/{pid}.{ext}.gz", timeout=60) as r:
                b = r.read()
            if len(b) > 500: p = RAW/f"{pid}.{ext}.gz"; p.write_bytes(b); return pid, p
        except Exception: continue
    return pid, None

t0=time.time()
with ThreadPoolExecutor(max_workers=16) as ex: got = list(ex.map(fetch, need))
paths = {p: f for p, f in got if f}
print(f"fetched {len(paths):,}/{len(need):,} in {(time.time()-t0)/60:.1f} min")

## Cell 4: Parse apo chains, build labels -- NEW: also extract per-residue CA B-factor, per-chain z-scored

In [ ]:
AA3_TO_1={"ALA":"A","ARG":"R","ASN":"N","ASP":"D","CYS":"C","GLN":"Q","GLU":"E","GLY":"G",
          "HIS":"H","ILE":"I","LEU":"L","LYS":"K","MET":"M","PHE":"F","PRO":"P","SER":"S",
          "THR":"T","TRP":"W","TYR":"Y","VAL":"V","MSE":"M"}
MIN_RES, MAX_RES = 30, 1500

def parse_unit(u):
    pid, cid = u["apo_id"], u["apo_chain"]
    if pid not in paths: return None
    path = paths[pid]
    try:
        parser = MMCIFParser(QUIET=True) if ".cif" in path.name else PDBParser(QUIET=True, PERMISSIVE=True)
        with gzip.open(path, "rt") as fh:
            model = next(parser.get_structure(pid, fh).get_models())
        ch = model[cid]
    except Exception:
        return None
    residues = [r for r in ch if r.get_resname().strip().upper() in AA3_TO_1 and "CA" in r]
    if not (MIN_RES <= len(residues) <= MAX_RES): return None
    seq, ca, per_res, resnums, bfac = [], [], [], [], []
    for r in residues:
        seq.append(AA3_TO_1.get(r.get_resname().strip().upper(), "X"))
        ca.append(r["CA"].coord)
        bfac.append(r["CA"].get_bfactor())
        atoms = [a.coord for a in r.get_unpacked_list() if a.element not in ("H","D")]
        per_res.append(np.asarray(atoms, dtype=np.float32).reshape(-1,3))
        resnums.append(str(r.id[1]) + r.id[2].strip())   # e.g. "60" or "60A" (insertion code)
    y = np.array([1.0 if rn in u["pocket_resnums"] else 0.0 for rn in resnums], dtype=np.float32)
    n_found = int(y.sum())
    if n_found < 3: return None   # too few pocket residues actually present in this chain -- skip
    # any-pocket aux label: cryptic OR noncryptic (union) -- a strictly
    # easier/broader task than the main cryptic-only label
    any_pocket_resnums = u["pocket_resnums"] | u["noncryptic_resnums"]
    y_aux = np.array([1.0 if rn in any_pocket_resnums else 0.0 for rn in resnums], dtype=np.float32)
    bfac = np.asarray(bfac, dtype=np.float32)
    bfac_z = (bfac - bfac.mean()) / (bfac.std() + 1e-6)   # per-chain relative flexibility
    return dict(pdb_id=pid, chain_id=cid, seq="".join(seq), bfac=bfac_z,
                ca=np.asarray(ca, dtype=np.float32), per_res=per_res, y=y, y_aux=y_aux,
                split=u["split"], pRMSD=u["pRMSD"], n_pocket_labeled=len(u["pocket_resnums"]),
                n_pocket_found=n_found)

from joblib import Parallel, delayed
t0 = time.time()
parsed = Parallel(n_jobs=max(1, os.cpu_count() or 2), verbose=1)(
    delayed(parse_unit)(u) for u in units)
records = [r for r in parsed if r is not None]
print(f"\nparsed in {(time.time()-t0)/60:.1f} min")
print(f"usable chains: {len(records):,}/{len(units):,}")
n_res = sum(len(r["y"]) for r in records)
n_pos = sum(int(r["y"].sum()) for r in records)
n_pos_aux = sum(int(r["y_aux"].sum()) for r in records)
print(f"residues {n_res:,} | cryptic-pocket residues {n_pos:,} ({100*n_pos/max(n_res,1):.2f}%) | "
      f"any-pocket (aux) residues {n_pos_aux:,} ({100*n_pos_aux/max(n_res,1):.2f}%)")
for s in ("train","test"):
    rs = [r for r in records if r["split"] == s]
    print(f"  {s:<6} {len(rs):>5} chains")

## Cell 5: Featuriser definitions (identical to Phase117)

In [ ]:
MAX_HOM_DIM = 2
BETTI_RADII = (2.0, 3.0, 5.0)
NBHD_RADIUS = 12.0
MAX_NBHD_ATOMS = 300
FEATS_PER_DIM = 6 + 2 + len(BETTI_RADII)
TOPO_DIM = FEATS_PER_DIM * (MAX_HOM_DIM + 1) + 2

TOPO_NAMES = [f"H{d}_{s}" for d in range(MAX_HOM_DIM + 1)
              for s in ["n_bars","total_pers","max_pers","mean_pers","std_pers","entropy",
                        "mean_birth","mean_death", *[f"betti_r{r:g}" for r in BETTI_RADII]]] \
             + ["nbhd_n_atoms", "nbhd_gyration"]
BURIAL_NAMES = ["nbhd_n_atoms", "nbhd_gyration"]


def _diagrams(pts):
    pts = np.ascontiguousarray(pts, dtype=np.float64)
    st, squared = None, True
    if len(pts) >= 5:
        try:
            st = gudhi.AlphaComplex(points=pts).create_simplex_tree()
        except Exception:
            st = None
    if st is None:
        st = gudhi.RipsComplex(points=pts, max_edge_length=float(NBHD_RADIUS)) \
                  .create_simplex_tree(max_dimension=MAX_HOM_DIM + 1)
        squared = False
    st.compute_persistence(homology_coeff_field=2, persistence_dim_max=True)
    out = {}
    for d in range(MAX_HOM_DIM + 1):
        b = np.asarray(st.persistence_intervals_in_dimension(d), dtype=np.float64).reshape(-1, 2)
        out[d] = np.sqrt(np.maximum(b, 0.0)) if (squared and len(b)) else b   # alpha -> angstrom
    return out


def _vectorize(dgms, pts):
    feats = []
    for d in range(MAX_HOM_DIM + 1):
        bars = dgms.get(d, np.zeros((0, 2)))
        fin = bars[np.isfinite(bars).all(axis=1)] if len(bars) else bars.reshape(-1, 2)
        if len(fin) == 0:
            feats.extend([0.0] * FEATS_PER_DIM); continue
        b, dth = fin[:, 0], fin[:, 1]
        pers = np.maximum(dth - b, 0.0); tot = float(pers.sum())
        if tot > 0:
            p = pers / tot; nz = p[p > 0]; ent = float(-(nz * np.log(nz)).sum())
        else:
            ent = 0.0
        feats.extend([float(len(fin)), tot, float(pers.max()), float(pers.mean()),
                      float(pers.std()), ent, float(b.mean()), float(dth.mean()),
                      *[float(((b <= r) & (dth > r)).sum()) for r in BETTI_RADII]])
    c = pts - pts.mean(0, keepdims=True)
    feats.extend([float(len(pts)), float(np.sqrt((c ** 2).sum(1).mean())) if len(pts) else 0.0])
    return np.asarray(feats, dtype=np.float32)


def topo_for_chain(ca_xyz, per_res):
    atoms = np.concatenate([a for a in per_res if len(a)], axis=0)
    tree = cKDTree(atoms)
    out = np.zeros((len(ca_xyz), TOPO_DIM), np.float32)
    for i, ca in enumerate(ca_xyz):
        idx = tree.query_ball_point(ca, NBHD_RADIUS)
        if len(idx) > MAX_NBHD_ATOMS:
            _, idx = tree.query(ca, k=MAX_NBHD_ATOMS)
        pts = atoms[np.asarray(idx, dtype=int)]
        if len(pts) >= 4:
            out[i] = _vectorize(_diagrams(pts), pts)
    return out

## Cell 6: Compute topology (identical to Phase117)

In [ ]:
n_res_total = sum(len(r["ca"]) for r in records)
print(f"computing persistent homology on {n_res_total:,} residues "
      f"({os.cpu_count()} workers, ~6.4 ms/residue -> ETA {n_res_total*0.0064/60:.0f} min)")
t0 = time.time()
topo_all = Parallel(n_jobs=max(1, os.cpu_count() or 2), verbose=5)(
    delayed(topo_for_chain)(r["ca"], r["per_res"]) for r in records)
el = time.time() - t0
print(f"\ndone in {el/60:.1f} min ({1000*el/max(n_res_total,1):.2f} ms/residue)")

topo_cat = np.concatenate(topo_all, axis=0)
keep = topo_cat.std(axis=0) > 1e-9
dropped = [TOPO_NAMES[i] for i in range(TOPO_DIM) if not keep[i]]
names = [TOPO_NAMES[i] for i in range(TOPO_DIM) if keep[i]]
print(f"pruned {len(dropped)} constant column(s): {dropped}")
mu, sd = topo_cat[:, keep].mean(0), topo_cat[:, keep].std(0); sd[sd < 1e-6] = 1.0

g = torch.Generator().manual_seed(0)
Q, _ = torch.linalg.qr(torch.randn(3, 3, generator=g))
Q = (Q * torch.sign(torch.det(Q))).numpy().astype(np.float32)
shift = np.array([31.7, -12.4, 88.1], dtype=np.float32)
ca0, pr0 = records[0]["ca"][:40], records[0]["per_res"][:40]
f_ref = topo_for_chain(ca0, pr0)
f_rot = topo_for_chain(ca0 @ Q.T + shift, [a @ Q.T + shift for a in pr0])
rel = float((np.abs(f_ref - f_rot) / np.maximum(np.abs(f_ref).max(0), 1e-6)).max())
print(f"rotation invariance: {rel:.2e}")
assert rel < 1e-3

## Cell 7: Recompute backbone dihedrals (identical to Phase117)

In [ ]:
AA3 = set(AA3_TO_1)

def dihedrals_for(pid, path, cid, n_expect):
    try:
        parser = MMCIFParser(QUIET=True) if ".cif" in path.name else PDBParser(QUIET=True, PERMISSIVE=True)
        with gzip.open(path,"rt") as fh:
            model = next(parser.get_structure(pid, fh).get_models())
        ch = model[cid]
    except Exception:
        return None
    residues=[r for r in ch if r.get_resname().strip().upper() in AA3 and "CA" in r]
    if len(residues) != n_expect:
        return None
    ppmap={}
    for pp in PPBuilder().build_peptides(ch, aa_only=False):
        for res,(phi,psi) in zip(pp, pp.get_phi_psi_list()):
            ppmap[res.id]=(phi,psi)
    phis,psis=[],[]
    for r in residues:
        phi,psi = ppmap.get(r.id, (None,None))
        phis.append(phi); psis.append(psi)
    def enc(angles):
        v=torch.tensor([a if a is not None else 0.0 for a in angles], dtype=torch.float32)
        m=torch.tensor([a is not None for a in angles], dtype=torch.float32)
        return torch.stack([torch.sin(v)*m, torch.cos(v)*m],dim=1), m.unsqueeze(1)
    pf,pm = enc(phis); sf,sm = enc(psis)
    return torch.cat([pf,sf,pm,sm],dim=1)

t0=time.time()
dihed = Parallel(n_jobs=max(1, os.cpu_count() or 2), verbose=1)(
    delayed(dihedrals_for)(r["pdb_id"], paths[r["pdb_id"]], r["chain_id"], len(r["ca"]))
    for r in records)
n_ok = sum(1 for x in dihed if x is not None)
print(f"\ndihedrals recovered for {n_ok:,}/{len(records):,} chains in {(time.time()-t0)/60:.1f} min")

## Cell 8: ESM2 embeddings (35M, identical to Phase117)

In [ ]:
from transformers import AutoTokenizer, AutoModel
ESM_NAME = "facebook/esm2_t12_35M_UR50D"
tok = AutoTokenizer.from_pretrained(ESM_NAME)
esm = AutoModel.from_pretrained(ESM_NAME).to(DEVICE).eval()
ESM_DIM = esm.config.hidden_size
print(f"{ESM_NAME}: hidden {ESM_DIM}, device {DEVICE}")

@torch.no_grad()
def embed_batch(seqs, max_len=1502):
    enc = tok(seqs, return_tensors="pt", padding=True, truncation=True, max_length=max_len)
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    out = esm(**enc).last_hidden_state
    embs = []
    for i, s in enumerate(seqs):
        n = len(s)
        embs.append(out[i, 1:1+n, :].float().cpu())
    return embs

order = sorted(range(len(records)), key=lambda i: len(records[i]["seq"]))
BATCH = 16
t0=time.time()
esm_feats = [None]*len(records)
for bi in range(0, len(order), BATCH):
    idx = order[bi:bi+BATCH]
    seqs = [records[i]["seq"] for i in idx]
    embs = embed_batch(seqs)
    for i, e in zip(idx, embs):
        esm_feats[i] = e
    if bi % (BATCH*20) == 0:
        print(f"  {bi}/{len(order)}  [{(time.time()-t0)/60:.1f}m]")

n_res = sum(len(r["ca"]) for r in records)
print(f"\nESM2 embeddings for {len(records):,} chains / {n_res:,} residues "
      f"in {(time.time()-t0)/60:.1f} min ({1000*(time.time()-t0)/max(n_res,1):.2f} ms/residue)")
del esm; torch.cuda.empty_cache()

## Cell 9: Assemble PyG graphs -- NEW: insert the B-factor column right after ESM, before topology; track BFAC_COL_IDX for the ablation

In [ ]:
AA_ALPHABET = "ACDEFGHIKLMNPQRSTVWYX"
AA_TO_IDX = {a: i for i, a in enumerate(AA_ALPHABET)}

def radius_graph_capped(pos, r, max_neighbors=32):
    n = pos.size(0)
    pos = pos - pos.mean(dim=0, keepdim=True)
    dist = torch.cdist(pos, pos).clone()
    dist.fill_diagonal_(float("inf"))
    mask = dist <= r
    if max_neighbors is not None and max_neighbors < n:
        _, knn = torch.topk(dist, max_neighbors, dim=1, largest=False)
        keep_ = torch.zeros_like(mask); keep_.scatter_(1, knn, True); mask &= keep_
    mask = mask | mask.t()
    tgt, src = mask.nonzero(as_tuple=True)
    return torch.stack([src, tgt], dim=0).long()

BFAC_COL_IDX = 21 + 6 + ESM_DIM   # index of the new B-factor column
N_PRE = 21 + 6 + ESM_DIM + 1        # +1 for B-factor, now part of the "pre-topology" block
dataset = []
n_dropped = 0
for i, r in enumerate(records):
    if dihed[i] is None or esm_feats[i] is None:
        n_dropped += 1; continue
    n = len(r["ca"])
    if dihed[i].shape[0] != n or esm_feats[i].shape[0] != n:
        n_dropped += 1; continue
    onehot = torch.zeros(n, len(AA_ALPHABET))
    onehot[torch.arange(n), torch.tensor([AA_TO_IDX.get(c, 20) for c in r["seq"]])] = 1.0
    topo_i = topo_all[i][:, keep]
    topo_z = torch.from_numpy(((topo_i - mu) / sd).astype(np.float32))
    bfac_col = torch.from_numpy(r["bfac"]).unsqueeze(1).float()
    x = torch.cat([onehot, dihed[i], esm_feats[i], bfac_col, topo_z], dim=1)

    pos = torch.from_numpy(r["ca"]); pos = pos - pos.mean(0, keepdim=True)
    ei = radius_graph_capped(pos, 8.0, 32)
    d_ = (pos[ei[0]]-pos[ei[1]]).norm(dim=1,keepdim=True)
    sep = (ei[0]-ei[1]).abs().float().unsqueeze(1)

    y_np = r["y"].astype(bool)
    true_center = torch.from_numpy(r["ca"][y_np].mean(axis=0) - r["ca"].mean(axis=0)) if y_np.any() else None
    if true_center is None:
        n_dropped += 1; continue

    d = Data(x=x, pos=pos, edge_index=ei, edge_attr=torch.cat([d_, torch.log1p(sep)], 1),
             y=torch.from_numpy(r["y"]), y_aux=torch.from_numpy(r["y_aux"]))
    d.true_center = true_center.float()
    d.pdb_id, d.chain_id, d.split = r["pdb_id"], r["chain_id"], r["split"]
    d.num_nodes = n
    assert torch.isfinite(d.x).all()
    dataset.append(d)

print(f"built {len(dataset):,} graphs ({n_dropped} dropped: alignment/labeling failures)")
n_res_ = sum(d.num_nodes for d in dataset); n_pos_ = sum(int(d.y.sum()) for d in dataset)
n_pos_aux_ = sum(int(d.y_aux.sum()) for d in dataset)
POS_WEIGHT = (n_res_ - n_pos_) / max(n_pos_, 1)
POS_WEIGHT_AUX = (n_res_ - n_pos_aux_) / max(n_pos_aux_, 1)
print(f"residues {n_res_:,} | cryptic positives {n_pos_:,} ({100*n_pos_/max(n_res_,1):.2f}%) "
      f"| pos_weight {POS_WEIGHT:.2f}")
print(f"any-pocket (aux) positives {n_pos_aux_:,} ({100*n_pos_aux_/max(n_res_,1):.2f}%) "
      f"| pos_weight_aux {POS_WEIGHT_AUX:.2f}")
for s in ("train","test"):
    ds = [d for d in dataset if d.split == s]
    print(f"  {s:<6} {len(ds):>5} chains")

## Cell 10: Carve val split, integrity checks, save (identical to Phase117 except output filename)

In [ ]:
rng = np.random.RandomState(42)
train_idx = [i for i,d in enumerate(dataset) if d.split == "train"]
perm = rng.permutation(len(train_idx))
n_val = max(1, int(0.08 * len(perm)))
val_set = {train_idx[i] for i in perm[:n_val]}
for i, d in enumerate(dataset):
    if i in val_set: d.split = "val"

for s in ("train","val","test"):
    ds = [d for d in dataset if d.split == s]
    r_ = sum(d.num_nodes for d in ds); p_ = sum(int(d.y.sum()) for d in ds)
    pa_ = sum(int(d.y_aux.sum()) for d in ds)
    print(f"{s:<6} chains {len(ds):>5}  residues {r_:>7,}  cryptic-pos {p_:>5} ({100*p_/max(r_,1):>5.2f}%)  "
          f"any-pocket-pos {pa_:>5} ({100*pa_/max(r_,1):>5.2f}%)")

for d in dataset:
    assert torch.isfinite(d.x).all() and torch.isfinite(d.pos).all()
    assert d.edge_index.max() < d.num_nodes and d.edge_index.min() >= 0
    assert (d.y_aux >= d.y).all(), "y_aux must be a superset of y (cryptic implies any-pocket)"
print("integrity OK (including y_aux >= y superset check)")

p = WORK / "phase120_cryptobench_graphs_bfactor.pt"
torch.save({"dataset": dataset, "n_pre_topo": N_PRE, "topo_dim": int(keep.sum()),
            "topo_names": names, "dropped_names": dropped,
            "topo_mean": mu, "topo_std": sd, "esm_dim": ESM_DIM,
            "pos_weight": POS_WEIGHT, "pos_weight_aux": POS_WEIGHT_AUX,
            "aa_alphabet": AA_ALPHABET,
            "bfac_col_idx": BFAC_COL_IDX, "source": "CryptoBench (Skrhak et al. 2025, btae745) + per-chain z-scored B-factor feature"}, p)
print(f"saved -> {p} ({p.stat().st_size/1e6:.0f} MB)")

## Cell 11: Model (identical to Phase118/119/121; imports needed here since fresh ingestion never needed them)

In [ ]:
from torch_geometric.loader import DataLoader
from torch_geometric.utils import softmax as pyg_softmax
from sklearn.metrics import average_precision_score, roc_auc_score

class EGCL(nn.Module):
    def __init__(s,h,ea=2,act=nn.SiLU,uc=True):
        super().__init__(); s.uc=uc
        s.e=nn.Sequential(nn.Linear(2*h+1+ea,h),act(),nn.Linear(h,h),act())
        s.n=nn.Sequential(nn.Linear(2*h,h),act(),nn.Linear(h,h))
        s.a=nn.Sequential(nn.Linear(h,1),nn.Sigmoid())
        if uc:
            s.c=nn.Sequential(nn.Linear(h,h),act(),nn.Linear(h,1,bias=False))
            nn.init.xavier_uniform_(s.c[-1].weight,gain=0.001)
        s.norm=nn.LayerNorm(h)
    def forward(s,h,x,ei,ea):
        src,dst=ei; rel=x[src]-x[dst]; d2=(rel**2).sum(-1,keepdim=True)
        m=s.e(torch.cat([h[src],h[dst],d2,ea],-1)); m=m*s.a(m)
        agg=torch.zeros_like(h).index_add_(0,dst,m)
        deg=torch.zeros(h.size(0),1,device=h.device).index_add_(0,dst,torch.ones_like(d2)).clamp(min=1)
        h=s.norm(h+s.n(torch.cat([h,agg/deg],-1)))
        if s.uc:
            w=s.c(m)*rel/(d2.sqrt()+1.0); x=x+torch.zeros_like(x).index_add_(0,dst,w)/deg
        return h,x

class Fus(nn.Module):
    def __init__(s,h,t):
        super().__init__(); s.p=nn.Linear(t,h); s.g=nn.Sequential(nn.Linear(h+t,h),nn.Sigmoid())
    def forward(s,h,t): return h+s.g(torch.cat([h,t],-1))*s.p(t)

class LocalizingEGNN(nn.Module):
    def __init__(s, n_pre, td, h=128, L=4, dr=0.1, use_loc_head=True):
        super().__init__(); s.n_pre=n_pre; s.use_loc_head=use_loc_head
        s.embed=nn.Sequential(nn.Linear(n_pre+td,h),nn.SiLU(),nn.Linear(h,h))
        s.ls=nn.ModuleList([EGCL(h,uc=(i<L-1)) for i in range(L)])
        s.fs=nn.ModuleList([Fus(h,td) for _ in range(L)])
        s.head=nn.Sequential(nn.Linear(h,h//2),nn.SiLU(),nn.Dropout(dr),nn.Linear(h//2,1))
        if use_loc_head:
            s.loc_head=nn.Sequential(nn.Linear(h,h//2),nn.SiLU(),nn.Linear(h//2,1))

    def forward(s, d):
        h=s.embed(d.x); pos=d.pos; t=d.x[:,s.n_pre:]
        for l,f in zip(s.ls,s.fs): h,pos=l(h,pos,d.edge_index,d.edge_attr); h=f(h,t)
        logits = s.head(h).squeeze(-1)
        pred_center = None
        if s.use_loc_head:
            batch = d.batch if hasattr(d, "batch") and d.batch is not None \
                    else torch.zeros(h.size(0), dtype=torch.long, device=h.device)
            loc_logit = s.loc_head(h).squeeze(-1)
            attn = pyg_softmax(loc_logit, batch)
            weighted = attn.unsqueeze(-1) * pos
            n_graphs = int(batch.max().item()) + 1
            pred_center = torch.zeros(n_graphs, 3, device=h.device).index_add_(0, batch, weighted)
        return logits, pred_center

GPU_OK=False
if torch.cuda.is_available():
    try:
        _p=torch.randn(256,256,device="cuda"); (_p@_p).sum().item(); torch.cuda.synchronize(); GPU_OK=True
        print(f"GPU: {torch.cuda.get_device_name(0)} (probe PASS)")
    except RuntimeError as e: print("GPU unusable:", e)
DEVICE = torch.device("cuda" if GPU_OK else "cpu")
splits = {s: [d for d in dataset if d.split == s] for s in ("train","val","test")}
# TOPO_DIM as a bare global stays at its PRE-filter value (35) throughout
# ingestion -- only Cell 10's SAVE line computes the correct POST-filter
# value (int(keep.sum())=34, after dropping near-constant columns) inline,
# without ever reassigning the variable itself. Phase118/119/121 never hit
# this because they reload TOPO_DIM from a saved blob (already correct);
# this combined ingest+train notebook needs the fix explicitly:
TOPO_DIM = int(keep.sum())
print(f"device {DEVICE} | node dim {dataset[0].x.size(1)} | N_PRE {N_PRE} | TOPO_DIM {TOPO_DIM} | BFAC_COL_IDX {BFAC_COL_IDX} | "
      f"pos_weight {POS_WEIGHT:.2f} | " + " | ".join(f"{k} {len(v)}" for k,v in splits.items()))
WORK = Path("/kaggle/working"); (WORK/"cache").mkdir(parents=True, exist_ok=True)

## Cell 12: Training machinery (identical pattern to Phase121)

In [ ]:
def run_epoch(model, loader, opt=None, lam=0.0):
    tr = opt is not None; model.train(tr)
    crit = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POS_WEIGHT], device=DEVICE))
    huber = nn.SmoothL1Loss(beta=4.0)
    ps, ys, cd = [], [], []
    for b in loader:
        b = b.to(DEVICE)
        with torch.set_grad_enabled(tr):
            logits, pred_c = model(b)
            loss = crit(logits, b.y)
            if lam > 0 and pred_c is not None:
                true_c = b.true_center.view(-1, 3)
                loss = loss + lam * huber(pred_c, true_c)
                cd.append((pred_c - true_c).norm(dim=1).detach().cpu().numpy())
        if tr:
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        ps.append(torch.sigmoid(logits).detach().cpu().numpy()); ys.append(b.y.cpu().numpy())
    p, y = np.concatenate(ps), np.concatenate(ys)
    out = {"pr": average_precision_score(y, p), "roc": roc_auc_score(y, p)}
    if cd:
        d_ = np.concatenate(cd)
        out["dcc"] = float((d_ < 4.0).mean()); out["mean_center_err"] = float(d_.mean())
    return out


def fit(use_loc_head, lam, seed, epochs=40, patience=8, batch=16):
    torch.manual_seed(seed); np.random.seed(seed)
    model = LocalizingEGNN(N_PRE, TOPO_DIM, use_loc_head=use_loc_head).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    tl = DataLoader(splits["train"], batch_size=batch, shuffle=True)
    vl = DataLoader(splits["val"], batch_size=batch)
    best, state, bad = -1, None, 0
    for ep in range(epochs):
        run_epoch(model, tl, opt, lam)
        v = run_epoch(model, vl, lam=lam)
        sch.step()
        if v["pr"] > best:
            best, state, bad = v["pr"], {k:t.clone() for k,t in model.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(state); model.eval()
    return model, best

## Cell 13: Train WITH B-factor first (data pristine), then zero the one B-factor column for WITHOUT, same seed=0, same run (same architecture both arms -- avoids the RNG-shift confound from Phase119)

In [ ]:
BEST_LAMBDA = 0.1
results = []
t0 = time.time()

m, val_pr = fit(use_loc_head=True, lam=BEST_LAMBDA, seed=0)
v = run_epoch(m, DataLoader(splits["val"], batch_size=16), lam=BEST_LAMBDA)
torch.save(m.state_dict(), WORK/"cache/cryptobench_withbfac_seed0.pt")
results.append(dict(arm="with B-factor", val_pr=val_pr, val_dcc=v.get("dcc", float("nan"))))
print(f"  with B-factor    seed0: val PR {val_pr:.4f}  val DCC {v.get('dcc', float('nan')):.4f}  [{(time.time()-t0)/60:.1f}m]")

for d in dataset:
    d.x[:, BFAC_COL_IDX] = 0.0   # zero JUST the B-factor column -- same architecture, isolates its contribution
m2, val_pr2 = fit(use_loc_head=True, lam=BEST_LAMBDA, seed=0)
v2 = run_epoch(m2, DataLoader(splits["val"], batch_size=16), lam=BEST_LAMBDA)
torch.save(m2.state_dict(), WORK/"cache/cryptobench_nobfac_seed0.pt")
results.append(dict(arm="without B-factor (zeroed)", val_pr=val_pr2, val_dcc=v2.get("dcc", float("nan"))))
print(f"  without B-factor seed0: val PR {val_pr2:.4f}  val DCC {v2.get('dcc', float('nan')):.4f}  [{(time.time()-t0)/60:.1f}m]")

print(f"\n--- B-factor ablation (single seed=0, same run) ---")
with_pr = results[0]["val_pr"]
for r in results:
    print(f"  {r['arm']:<30} val PR {r['val_pr']:.4f}  ({r['val_pr']-with_pr:+.4f} vs with-B-factor)  val DCC {r['val_dcc']:.4f}")
json.dump(results, open(WORK/"phase120_bfactor_results.json", "w"), indent=2)
print("\nsaved -> phase120_bfactor_results.json")

---

## Reading this

- **with-B-factor clearly beats without** (gap outside typical
  single-seed noise) -- crystallographic flexibility carries real signal
  for cryptic-pocket prediction, worth an 8-seed confirmation.
- **Close/tied** -- B-factor doesn't add exploitable signal beyond what
  topology+sequence already capture.
- **without beats with** -- B-factor might be adding noise (crystal
  packing artifacts, resolution-dependent noise unrelated to true
  flexibility) rather than signal.